In [5]:
# Enables IPython autoreload (two magic commands, the second takes a numeric argument)
%load_ext autoreload
%autoreload 2

import logging
import os
import time

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s  - %(name)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.6f}".format)


logger.info("Notebook initialized")

2026-05-06 13:10:02,737  - __main__ - INFO - Notebook initialized


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## Final pipeline using `src/data.py`

The above cells walked through the exploration step-by-step. The same pipeline is now packaged into two functions in `src/data.py`. The cells below verify that the imported functions produce the same output as the inline exploration.

In [6]:
from src.data import compute_returns, download_prices

prices = download_prices()
returns = compute_returns(prices)

print("Daily prices shape:", prices.shape)
print("Monthly returns shape:", returns.shape)
print("\nFirst 3 returns:")
print(returns.head(3))
print("\nLast 3 returns:")
print(returns.tail(3))

2026-05-06 13:10:02,756  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-06 13:10:02,762  - src.data - INFO - Computed monthly returns: (120, 8)


Daily prices shape: (2538, 8)
Monthly returns shape: (120, 8)

First 3 returns:
                 SPY      GOVT     EEMV       CME       BR      CBOE  \
date                                                                   
2015-01-31 -0.029629  0.029423 0.011831 -0.037789 0.039194  0.016556   
2015-02-28  0.056205 -0.017439 0.025305  0.124619 0.109189 -0.065708   
2015-03-31 -0.015745  0.006021 0.004426 -0.007544 0.038856 -0.043728   

                 ICE       ACN  
date                            
2015-01-31 -0.061836 -0.059120  
2015-02-28  0.144024  0.071403  
2015-03-31 -0.006068  0.040653  

Last 3 returns:
                 SPY      GOVT      EEMV      CME        BR      CBOE  \
date                                                                    
2024-10-31 -0.008924 -0.024286 -0.037161 0.021346 -0.019393  0.042466   
2024-11-30  0.059634  0.008489 -0.008945 0.056088  0.119321  0.013626   
2024-12-31 -0.024100  0.006945 -0.007585 0.004852 -0.038463 -0.094742   

           

In [7]:
from src.data import compute_returns, download_prices
from src.stats import arithmetic_mean, geometric_mean

prices = download_prices()
returns = compute_returns(prices)

print("Arithmetic mean (monthly):")
print(arithmetic_mean(returns))
print("\nGeometric mean (monthly):")
print(geometric_mean(returns))

2026-05-06 13:10:02,781  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet


2026-05-06 13:10:02,786  - src.data - INFO - Computed monthly returns: (120, 8)


Arithmetic mean (monthly):
SPY    0.011216
GOVT   0.000917
EEMV   0.002965
CME    0.012852
BR     0.016831
CBOE   0.012521
ICE    0.013060
ACN    0.015002
dtype: float64

Geometric mean (monthly):
SPY    0.010243
GOVT   0.000815
EEMV   0.002326
CME    0.011446
BR     0.014829
CBOE   0.010574
ICE    0.011327
ACN    0.012883
dtype: float64


In [8]:
from src.data import compute_returns, download_prices
from src.sharpe import tangency_portfolio_grid, tangency_portfolio_qp
from src.stats import arithmetic_mean, covariance_matrix

prices = download_prices()
returns = compute_returns(prices)
mu = arithmetic_mean(returns)
sigma = covariance_matrix(returns)

w_grid = tangency_portfolio_grid(mu, sigma, risk_free_rate=0.0)
w_qp = tangency_portfolio_qp(mu, sigma, risk_free_rate=0.0)

print("Grid:", w_grid.attrs)
print("QP:  ", w_qp.attrs)
print(f"\nWeight max diff: {(w_grid - w_qp).abs().max():.6f}")
print(f"Sharpe diff:     {abs(w_grid.attrs['sharpe'] - w_qp.attrs['sharpe']):.6f}")

2026-05-06 13:10:02,807  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-06 13:10:02,812  - src.data - INFO - Computed monthly returns: (120, 8)
2026-05-06 13:10:02,851  - src.optimizer - INFO - cvxpy MVO converged: variance=0.000185, return=0.001851, status=optimal
2026-05-06 13:10:02,855  - src.optimizer - INFO - cvxpy MVO converged: variance=0.000185, return=0.001851, status=optimal
2026-05-06 13:10:02,859  - src.optimizer - INFO - cvxpy MVO converged: variance=0.000185, return=0.001851, status=optimal
2026-05-06 13:10:02,863  - src.optimizer - INFO - cvxpy MVO converged: variance=0.000185, return=0.001851, status=optimal
2026-05-06 13:10:02,866  - src.optimizer - INFO - cvxpy MVO converged: variance=0.000185, return=0.001851, status=optimal
2026-05-06 13:10:02,871  - src.optimizer - INFO - cvxpy MVO converged: variance=0.000185, return=0.001851, status=optimal
2026-05-06 13:10:02,874  - src.opti

Grid: {'sharpe': 0.32406536226127514, 'return': 0.011731963362623376, 'volatility': 0.0362024601480382, 'method': 'grid'}
QP:   {'sharpe': 0.3240657888632708, 'return': 0.01168424990334827, 'volatility': 0.0360551786238629, 'method': 'qp'}

Weight max diff: 0.003623
Sharpe diff:     0.000000
